In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the set of all integers from `start` up to and including `stop`.

In [ ]:
let range (start : int) (stop : int) : int RSet.t =
  RSet.of_list (List.init (stop - start + 1) (fun i -> start + i))

## Sudoku

We use the function `solve` from the notebook `07-Davis-Putnam-JW.ipynb`.  This notebook imports `04-CNF.ipynb`, which defines the types `literal`, `clause`, and `cnf`.

In [ ]:
import_notebook "07-Davis-Putnam-JW.ipynb"

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is shown below.
<img src="sudoku.png" alt="A game of Sudoku" width="50%">

The function `create_puzzle` takes the argument `()`.  It returns the Sudoku as a list of rows, where every row is a list of 9 cells.  A cell that contains the digit `d` is represented as `Some d`, a cell that is left unspecified is represented as `None`.

**Implementation:** In order to make the puzzle easy to read, every row is written as a string that contains a `'*'` for every unspecified cell.  The function `cell` converts a character into a value of type `int option`.  `Char.code` returns the code of a character.  As the codes of the digits are consecutive, `Char.code ch - Char.code '0'` is the value of the digit `ch`.  `List.of_seq (String.to_seq row)` converts a string into the list of its characters.

In [ ]:
let create_puzzle () : int option list list =
  let rows = ["8********";
              "**36*****";
              "*7**9*2**";
              "*5***7***";
              "****457**";
              "***1***3*";
              "**1****68";
              "**85***1*";
              "*9****4**"] in
  let cell (ch : char) : int option =
    if ch = '*' then None else Some (Char.code ch - Char.code '0')
  in
  List.map (fun row -> List.map cell (List.of_seq (String.to_seq row))) rows

In [ ]:
create_puzzle ()

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.

The function `var_name` takes three arguments:
- `row` is the number of a row,
- `col` is the number of a column,
- `digit` is a digit.

It returns the string `Q<row,col,digit>`, which is interpreted as a variable name.

In [ ]:
let var_name (row : int) (col : int) (digit : int) : string =
  Printf.sprintf "Q<%d,%d,%d>" row col digit

In [ ]:
var_name 1 2 3

The function `at_most_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of `s` is true.
$$ \texttt{at\_most\_one}(S) = \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q\rangle \in S \times S \wedge p \not= q\bigr\}. $$

In [ ]:
let at_most_one (s : string RSet.t) : cnf =
  RSet.flat_map
    (fun p -> RSet.filter_map (fun q -> if p <> q then Some (RSet.of_list [Neg p; Neg q]) else None) s)
    s

The function `at_least_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of `s` is true.  This set contains just one clause, namely the clause consisting of the variables of `s`.

In [ ]:
let at_least_one (s : string RSet.t) : cnf =
  RSet.singleton (RSet.map (fun p -> Pos p) s)

The function `exactly_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables of `s` is true.

In [ ]:
let exactly_one (s : string RSet.t) : cnf =
  RSet.union (at_most_one s) (at_least_one s)

In [ ]:
exactly_one (RSet.of_list ["a"; "b"; "c"])

A `position` is a pair of coordinates of the form `(row, col)` where
`row` specifies the row and `col` specifies the column.

In [ ]:
type position = int * int

The function `exactly_one_for_digit` takes two arguments:
* `l` is a set of positions of a Sudoku.  These are pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
* `digit` is an element from the set $\{1,\cdots,9\}$.

It returns a set of clauses expressing the fact that the given `digit`
appears exactly once among the cells specified by the positions in `l`.

In [ ]:
let exactly_one_for_digit (l : position RSet.t) (digit : int) : cnf =
  exactly_one (RSet.map (fun (row, col) -> var_name row col digit) l)

The function `exactly_one_for_every_digit` takes one argument:
- `l` is a set of positions of a Sudoku.

It returns a set of clauses expressing that all of the nine digits occur at exactly one position in `l`.  The partial application `exactly_one_for_digit l` maps a digit `d` to the clauses stating that `d` occurs exactly once in `l`.

In [ ]:
let exactly_one_for_every_digit (l : position RSet.t) : cnf =
  RSet.flat_map (exactly_one_for_digit l) (range 1 9)

The function `exactly_one_digit` takes two arguments:
- `row` is the number of a row,
- `col` is the number of a column.

These specify a field of the Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [ ]:
let exactly_one_digit (row : int) (col : int) : cnf =
  exactly_one (RSet.map (var_name row col) (range 1 9))

The function `constraints_from_puzzle` takes the argument `()`.  It returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.  Below, the function `prettify` from the notebook `04-CNF.ipynb` is used to print these unit clauses.

**Implementation:** `List.nth l k` returns the element of the list `l` at position `k`, where the first element has the position 0.  For every position `(r, c)` whose cell contains a digit `d`, the unit clause $\{\texttt{Q<}r\texttt{,}c\texttt{,}d\texttt{>}\}$ is created.

In [ ]:
let constraints_from_puzzle () : cnf =
  let puzzle  = create_puzzle () in
  let indices = range 1 9 in
  RSet.filter_map
    (fun (r, c) ->
       match List.nth (List.nth puzzle (r - 1)) (c - 1) with
       | Some d -> Some (RSet.singleton (Pos (var_name r c d)))
       | None   -> None)
    (RSet.cartesian_product indices indices)

In [ ]:
RSet.iter (fun c -> print_endline (prettify (RSet.singleton c))) (constraints_from_puzzle ());
RSet.cardinal (constraints_from_puzzle ())

The function `get_block_cells` calculates the absolute grid coordinates for all 9 cells
within a specific 3x3 Sudoku block.  The Sudoku board is divided into a 3x3 grid of blocks.
This function takes the zero-based indices of a block and maps its local cells to the
1-based coordinates used by the solver variables.  It takes two arguments:
 * `r` is the row index of the 3x3 block (0, 1, or 2),
 * `c` is the column index of the 3x3 block (0, 1, or 2).

The function returns the set of the 9 positions `(row, col)` of the cells of this block on the full 9x9 Sudoku board.

For example, the set of coordinates for the top-middle block (`r = 0, c = 1`)
is given as follows:
```
get_block_cells 0 1 = {(1, 4), (1, 5), (1, 6),
                       (2, 4), (2, 5), (2, 6),
                       (3, 4), (3, 5), (3, 6)}
```

In [ ]:
let get_block_cells (r : int) (c : int) : position RSet.t =
  RSet.map (fun (row, col) -> (r * 3 + row, c * 3 + col))
           (RSet.cartesian_product (range 1 3) (range 1 3))

In [ ]:
get_block_cells 0 1

The function `all_constraints` takes the argument `()`.  It returns a set of clauses that encodes the given Sudoku.

In [ ]:
let all_constraints () : cnf =
  let digits = range 1 9 in
  RSet.unions [
    (* 1. Start with the constraints from the puzzle *)
    constraints_from_puzzle ();
    (* 2. There is exactly one digit in every field *)
    RSet.flat_map (fun (row, col) -> exactly_one_digit row col)
                  (RSet.cartesian_product digits digits);
    (* 3. All entries in a row are unique *)
    RSet.flat_map
      (fun row -> exactly_one_for_every_digit (RSet.map (fun col -> (row, col)) digits))
      digits;
    (* 4. All entries in a column are unique *)
    RSet.flat_map
      (fun col -> exactly_one_for_every_digit (RSet.map (fun row -> (row, col)) digits))
      digits;
    (* 5. All entries in a 3x3 square are unique *)
    RSet.flat_map (fun (r, c) -> exactly_one_for_every_digit (get_block_cells r c))
                  (RSet.cartesian_product (range 0 2) (range 0 2))
  ]

In [ ]:
RSet.cardinal (all_constraints ())

The function `sudoku` takes the argument `()`.  It solves the Sudoku.  The result has the type `cnf option`:
- If the Sudoku is solvable, the result is `Some solution`, where `solution` is a set of unit clauses.
- Otherwise, a message is printed and the result is `None`.

In [ ]:
let sudoku () : cnf option =
  let clauses  = all_constraints () in
  let solution = solve clauses in
  if not (RSet.mem RSet.empty solution) then
    Some solution
  else begin
    print_endline "The problem is not solvable!";
    None
  end

On my Mac Studio from 2022, this takes about 50 seconds.

In [ ]:
let solution = time "sudoku" sudoku

## Graphical Representation

The function `transform_solution` takes one argument:
- `solution` is a set of unit clauses representing a solution of the Sudoku.

It returns an association list that maps every position `(row, col)` to the digit in this cell.

**Implementation:** The function `first` from the notebook `07-Davis-Putnam-JW.ipynb` returns the only element of a unit clause.  `Scanf.sscanf s format f` reads the values described by `format` from the string `s` and passes them to the function `f`.  Here, the format `"Q<%d,%d,%d>"` extracts the row, the column, and the digit from a variable name.

In [ ]:
let transform_solution (solution : cnf) : (position * int) list =
  RSet.fold
    (fun unit_clause result ->
       match first unit_clause with
       | Pos var -> Scanf.sscanf var "Q<%d,%d,%d>" (fun row col digit -> ((row, col), digit) :: result)
       | Neg _   -> result)
    solution []

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which provides the function `Jupyter_notebook.display`.

In [ ]:
#require "jupyter.notebook";;

The function `show_solution` takes two arguments:
- `solution` is a set of unit clauses representing a solution of the Sudoku,
- `width` is the width of the table as a percentage of the width of the notebook.  It is an optional argument with the default value `"50%"`.

It displays the solution as an HTML table.  The digits that are given in the puzzle are shown in bold face.

In [ ]:
let show_solution ?(width = "50%") (solution : cnf) : unit =
  let solution_map = transform_solution solution in
  let puzzle = create_puzzle () in
  let html = Buffer.create 8192 in
  Buffer.add_string html
    (Printf.sprintf "<table style=\"width:%s; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;\">" width);
  for row = 0 to 8 do
    Buffer.add_string html "<tr>";
    for col = 0 to 8 do
      let (value, cell_style) =
        match List.nth (List.nth puzzle row) col with
        | Some d -> (Some d, "font-weight: bold;")
        | None   -> (List.assoc_opt (row + 1, col + 1) solution_map, "font-weight: normal;")
      in
      let is_gray  = (row / 3 + col / 3) mod 2 <> 0 in
      let bg_color = if is_gray then "#f0f0f0" else "#ffffff" in
      let border_style =
        "border: 1px solid #ccc;"
        ^ (if (col + 1) mod 3 = 0 && col < 8 then "border-right: 2px solid black;" else "")
        ^ (if (row + 1) mod 3 = 0 && row < 8 then "border-bottom: 2px solid black;" else "")
      in
      let text = match value with Some d -> string_of_int d | None -> "" in
      Buffer.add_string html
        (Printf.sprintf "<td style=\"%s width:30px; height:30px; text-align:center; font-size:20px; background-color:%s; %s\">%s</td>"
           border_style bg_color cell_style text)
    done;
    Buffer.add_string html "</tr>"
  done;
  Buffer.add_string html "</table>";
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

As `solution` has the type `cnf option`, we have to distinguish the two cases.

In [ ]:
match solution with
| Some s -> show_solution s
| None   -> print_endline "There is no solution."